# 🎭 Tiny Shakespeare Chat

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/tiny-shakespeare-chat/training/notebook.ipynb)

A GPT written **from scratch** (character-level, nanoGPT-style, 6 layers / 6 heads / 384-d, 10.75M
parameters) is trained in two stages on [Tiny Shakespeare](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare):
**A** next-character pretraining on the raw play, then **B** chat-tuning on 7,096 consecutive
dialogue-line pairs written as `<|user|> line <|end|> <|bot|> next line <|end|>`. The result answers a
message in play-style verse. It is served by the [Space](https://huggingface.co/spaces/shalev396/tiny-shakespeare-chat)
from the [model repo](https://huggingface.co/shalev396/tiny-shakespeare-chat).

All logic lives in `src/` (and `../model/model.py` for the network, tokenizer and sampler); this notebook
calls it step by step.

1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "tiny-shakespeare-chat"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: importing `src` puts ../model on sys.path, so `import model` is model/model.py
%matplotlib inline
import json

import pandas as pd
import torch
from IPython.display import Image as ShowImage, display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config
import model as M

device = torch.device(device)
print(utils.lib_versions("torch", "huggingface_hub", "safetensors", "numpy"), "| device:", device)

## 2. Config
Every hyperparameter lives in `src/config.py`; the defaults reproduce the deployed run (5,000 pretrain +
2,000 chat-tune iterations, batch 64 x 256 characters). `SMOKE_TEST=1` trains a 2-layer / 64-d model for
200 + 60 iterations on the same data (about a minute on CPU) and exports to `outputs/smoke/model/`.
A full run exports to `../model/` and **overwrites the deployed weights**. It took 541 s on a CUDA GPU;
on a CPU expect a few hours.

In [ ]:
# "0" = full run (GPU recommended; OVERWRITES ../model), "1" = smoke run.
# setdefault keeps a value set outside (e.g. `SMOKE_TEST=1 jupyter nbconvert --execute ...`).
os.environ.setdefault("SMOKE_TEST", "0")
PUSH_TO_HUB = False  # upload the exported model repo to the Hub at the end (section 8)

cfg = Config()
utils.set_seeds(cfg.seed, "torch")
print("export to:", cfg.model_dir)
cfg

## 3. Data
The corpus (1.1 MB) is downloaded once into `data/` (GitHub raw URL, with the byte-identical TensorFlow
tutorial copy as fallback). The tokenizer is the sorted set of its 65 characters plus the 3 chat markers,
each a single id (68 in total).
- **Stage A split:** the raw text, last 10 % = validation.
- **Stage B split:** the play is parsed into consecutive `(speaker, utterance)` turns; each neighbouring
  pair becomes one chat sample, in two variants (plain reply, and reply prefixed with the speaker name).
  The samples are shuffled (seed 42) and the last 5 % of the stream is validation.

In [ ]:
text = data_setup.download_corpus(cfg)
tok = data_setup.build_tokenizer(text)
text_train, text_val = data_setup.make_pretrain_split(cfg, tok, text)
chat_train, chat_val, n_pairs = data_setup.make_chat_split(cfg, tok, text)

print(f"{len(text):,} characters | vocab {tok.vocab_size} = {len(tok.chars)} chars + {tok.specials}")
display(pd.DataFrame({"stage A: raw play": {"train tokens": len(text_train), "val tokens": len(text_val)},
                      "stage B: chat pairs": {"train tokens": len(chat_train), "val tokens": len(chat_val)}}).T)
turns = data_setup.parse_dialogue(text)
print(f"{len(turns):,} dialogue turns -> {n_pairs:,} (line -> next line) pairs\n")
print(text[:300])
print("---- one chat-formatted training sample ----")
print(data_setup.build_chat_samples(turns, cfg.seed)[0][:400])

## 4. Load model
`model_builder.build_gpt` creates `model.GPT` from `../model/model.py`: token + learned position
embeddings, pre-LayerNorm transformer blocks (causal SDPA attention, 4x GELU MLP, dropout) and an LM head
tied to the token embedding (counted once). The vocabulary and `block_size` are constructor arguments, so
`save_pretrained` writes them to `config.json`. `load_deployed` loads the model that is deployed right now
(for the comparison in section 6).

In [ ]:
net = model_builder.build_gpt(cfg, tok).to(device)
print(f"GPT {cfg.n_layer}L/{cfg.n_head}H/{cfg.n_embd}d, block {cfg.block_size}: {utils.count_params(net):,} parameters")
deployed_net, deployed_info = model_builder.load_deployed(device)
if deployed_net is not None:
    print(f"deployed model: {utils.count_params(deployed_net):,} parameters | {deployed_info.get('source')}")

## 5. Training
Both stages use AdamW (weight decay 0.1 on matrices only, betas 0.9/0.95), linear warmup then cosine
decay, gradient clipping at 1.0 and mixed precision on CUDA. Train/val loss is *estimated* on random
batches every `eval_interval` iterations (the curves); section 6 scores the full validation splits.

**Stage A: pretrain** on the raw play (next-character prediction), peak LR 3e-4.

In [ ]:
hist_a = engine.train_stage(net, text_train, text_val, iters=cfg.max_iters, lr=cfg.lr, min_lr=cfg.min_lr,
                            warmup_iters=cfg.warmup_iters, cfg=cfg, device=device, stage="pretrain")
stage_a = model_builder.snapshot(net)          # frozen copy for the comparison in section 6
print(engine.continue_text(net, "ROMEO:", max_new_tokens=cfg.max_new_tokens, seed=cfg.seed))

**Stage B: chat-tune** the same network on the chat-formatted pairs, peak LR 1e-4. The model learns
that after `<|user|> ... <|end|> <|bot|>` comes a reply, closed by `<|end|>`, where generation stops.

In [ ]:
hist_b = engine.train_stage(net, chat_train, chat_val, iters=cfg.chat_iters, lr=cfg.chat_lr,
                            min_lr=cfg.chat_min_lr, warmup_iters=cfg.chat_warmup_iters, cfg=cfg,
                            device=device, stage="chat")
train_time_s = hist_a["time_s"] + hist_b["time_s"]
print(f"training time: {train_time_s:.0f} s")
curves = export.plot_history({"pretrain": hist_a, "chat": hist_b}, cfg.run_dir / "training_curves.png")
display(ShowImage(filename=str(curves)))

## 6. Evaluation
`engine.evaluate` is deterministic: the validation split is cut into non-overlapping `block_size` windows
and every character is scored once (cross-entropy in nats per character; bits per char = nats / ln 2;
perplexity = e^loss). Every variant is scored on both splits: a count-based **bigram** baseline, the
**stage A** snapshot, the **chat-tuned** model from this run and (when present) the **deployed** model.
Lower is better. (Smoke runs score only the first `eval_max_tokens` characters of each split.)

In [ ]:
def score(m):
    return (engine.evaluate(m, chat_val, device, cfg.eval_max_tokens),
            engine.evaluate(m, text_val, device, cfg.eval_max_tokens))

n = cfg.eval_max_tokens or None
bigram = {"chat_val_loss": model_builder.BigramLM(tok.vocab_size).fit(chat_train).loss(chat_val[:n]),
          "text_val_loss": model_builder.BigramLM(tok.vocab_size).fit(text_train).loss(text_val[:n])}
a_chat, a_text = score(stage_a.to(device))
chat_ev, text_ev = score(net)
run_name = f"stage A + B: chat-tuned GPT (this run, {cfg.n_layer}L/{cfg.n_embd}d)"
comparison = {
    "bigram baseline (counts, add-one)": bigram,
    f"stage A only: pretrained GPT, {cfg.max_iters:,} iters": {"chat_val_loss": a_chat["loss"], "text_val_loss": a_text["loss"]},
    run_name: {"chat_val_loss": chat_ev["loss"], "text_val_loss": text_ev["loss"]},
}
table = dict(comparison)
if deployed_net is not None:     # reference only: a full run replaces it in section 8
    d_chat, d_text = score(deployed_net)
    table["currently deployed model (../model)"] = {"chat_val_loss": d_chat["loss"], "text_val_loss": d_text["loss"]}
display(pd.DataFrame(table).T.round(4))
scores = export.scores(chat_ev, text_ev)
print({k: round(v, 4) for k, v in scores.items()})

In [ ]:
display(ShowImage(filename=str(export.plot_comparison(table, run_name, cfg.run_dir / "comparison.png"))))
by_position = {"chat val": chat_ev["loss_by_position"], "text val": text_ev["loss_by_position"]}
display(ShowImage(filename=str(export.plot_loss_by_position(by_position, cfg.run_dir / "loss_by_position.png"))))

## 7. Inference
Exactly what the Space and the Inference Endpoint run: the weights are staged in a folder, then
`model.load(dir, device).predict(message, history, max_new_tokens, temperature, top_k)` returns the reply
string (`stream(...)` yields it character by character for the chat UI). The messages come from
`../space/examples/messages.json`; the last cell calls `handler.EndpointHandler` like an endpoint request.

In [ ]:
staged = export.save_weights(net, cfg.run_dir / "staged")
predictor = M.load(staged, device.type)
examples = json.loads((utils.SPACE_DIR / "examples" / "messages.json").read_text(encoding="utf-8"))
for i, ex in enumerate(examples):
    reply = predictor.predict(ex["message"], ex["history"], max_new_tokens=ex["max_new_tokens"],
                              temperature=ex["temperature"], seed=i)
    context = f"  (history: {ex['history']})" if ex["history"] else ""
    print(f"USER: {ex['message']}{context}\nBOT:  {reply}\n")

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("handler", utils.MODEL_DIR / "handler.py")
handler = importlib.util.module_from_spec(spec)
spec.loader.exec_module(handler)
endpoint = handler.EndpointHandler(str(staged))
endpoint({"inputs": {"message": "Good morrow!", "history": []}, "parameters": {"max_new_tokens": 80, "seed": 0}})

## 8. Export
`export.export` writes this run into `cfg.model_dir` (`../model`, or `outputs/smoke/model` for smoke runs):
`model.safetensors` + `config.json`, `metrics.json` (validation metrics + the comparison of every variant
+ both training histories), the graphs in `assets/`, and refreshes the model card (YAML `model-index` and the
metrics table). Smoke exports also get copies of the card, `model.py`, `handler.py` and `requirements.txt`,
so the folder is a complete model repo. `PUSH_TO_HUB = True` uploads it (token from `hf auth login` or an
`HF_TOKEN` secret / env var).

In [ ]:
metrics = export.build_metrics(
    scores,
    model=export.model_name(net),
    data={"n_chars": len(text), "vocab_size": tok.vocab_size, "n_text_train": len(text_train),
          "n_text_val": len(text_val), "n_chat_pairs": n_pairs, "n_chat_train": len(chat_train),
          "n_chat_val": len(chat_val), "n_eval_tokens_chat": chat_ev["n_tokens"],
          "n_eval_tokens_text": text_ev["n_tokens"]},
    params=utils.count_params(net), train_time_s=train_time_s, device=device.type, smoke=cfg.smoke,
    source=f"retrained {utils.today()} with training/ ({device.type})", comparison=comparison,
    training={"pretrain": {"iters": cfg.max_iters, "est_train_loss": hist_a["train_loss"][-1],
                           "est_val_loss": hist_a["val_loss"][-1]},
              "chat": {"iters": cfg.chat_iters, "est_train_loss": hist_b["train_loss"][-1],
                       "est_val_loss": hist_b["val_loss"][-1]}},
)
export.export(net, cfg.model_dir, metrics,
              histories={"pretrain": {k: v for k, v in hist_a.items() if k != "time_s"},
                         "chat": {k: v for k, v in hist_b.items() if k != "time_s"}},
              loss_by_position=by_position)
print(M.load(cfg.model_dir, "cpu").predict("How fares the king?", seed=0))

for path in sorted(cfg.model_dir.rglob("*")):
    if path.is_file() and not {"__pycache__", ".git"} & set(path.parts):
        print(f"{path.relative_to(cfg.model_dir).as_posix():<28} {path.stat().st_size / 1e3:>10.1f} KB")

if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))